# kodoom: Gemma 3 27B on Kaggle TPU

Active stage: **free-text-gate** (project-plan.md: 1.2 step 1, v19).
Run all measures short/long prompts, runs a 3-helmo/4-typed-case preflight,
then the 40-helmo/20-typed-case gate with **gemma3-27b-tpu-bf16**.
It ends at **awaiting-review**. This is an experiment; model adoption awaits human review.

Use a fresh private Kaggle notebook, Internet enabled, and **TPU v5e-8**.
Accept Gemma's terms on Kaggle and add `google/gemma-3/flax/gemma3-27b-it/1`
under Inputs/Models. Python 3.11 through 3.13 and glibc 2.31+ are supported.
A public GitHub checkout needs no secret; a private checkout needs `GITHUB_TOKEN`.
The Flax model uses Kaggle access rather than an HF token.
See `docs/execution-workflow.md` for saving, restore and label review.


In [ ]:
PROVIDER = "kaggle"
PROFILE = "kaggle-tpu"
BACKEND = "jax"
CODE_REVISION = "1f1fe578153833ac23f365304b7c354815df13c7"
CHECKOUT = "/tmp/kodoom-tpu-code"
RUN_GATE = True  # False runs only the measured preflight
RESTORE_PREFLIGHT = ""  # optional attached 27B preflight ZIP
RESTORE_GATE = ""  # optional attached 27B gate ZIP
BASELINE_GATE_BUNDLE = ""  # optional 4B gate ZIP for verified paired review

## Bootstrap the pinned code


In [ ]:
import contextlib
import os
import types
import urllib.parse
import urllib.request

if PROVIDER == "kaggle":
    from kaggle_secrets import UserSecretsClient

    with contextlib.suppress(Exception):
        os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
elif PROVIDER == "colab":
    from google.colab import userdata

    with contextlib.suppress(Exception):
        os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
url = (
    "https://api.github.com/repos/Farahani1/kodum/contents/src/kodoom/runtime.py"
    + "?ref="
    + urllib.parse.quote(CODE_REVISION, safe="")
)
headers = {"Accept": "application/vnd.github.raw+json"}
if os.environ.get("GITHUB_TOKEN"):
    headers["Authorization"] = "Bearer " + os.environ["GITHUB_TOKEN"]
with urllib.request.urlopen(urllib.request.Request(url, headers=headers)) as response:
    source = response.read().decode("utf-8")
runtime = types.ModuleType("kodoom_bootstrap")
exec(compile(source, "runtime.py", "exec"), runtime.__dict__)
checkout = runtime.bootstrap(PROVIDER, CODE_REVISION, CHECKOUT, backend=BACKEND)

## Restore selected private artifacts


In [ ]:
from kodoom.workflow import restore_stage

if RESTORE_PREFLIGHT:
    restore_stage(PROVIDER, PROFILE, "preflight", RESTORE_PREFLIGHT)
if RESTORE_GATE:
    restore_stage(PROVIDER, PROFILE, "gate", RESTORE_GATE)

## TPU preflight

Input budgets, sharding, BF16, short/long warm timing and output integrity are checked first.


In [ ]:
from kodoom.workflow import run_current

preflight = run_current(provider=PROVIDER, profile=PROFILE, mode="preflight")

## Free-text gate


In [ ]:
gate = None
if RUN_GATE:
    gate = run_current(
        provider=PROVIDER,
        profile=PROFILE,
        mode="gate",
        model_revision=preflight["model_revision"],
    )

## Private output and human review

Review complete cases and option descriptions separately. For this experiment:
at most 1 meaning change in 40 helmo records and at most 10% needing edits;
at most 10% of typed cases needing edits. Do not adopt 27B or start full translation automatically.

Optional paired review requires the real 4B gate bundle with identical selected English inputs.
The balanced diagnostic sample cannot estimate dataset-wide label error rates or detector coverage.

Save a private notebook version with outputs. Attach its output in a fresh session
and restore the 27B ZIPs to verify persistence. Source-only saves do not prove durability.
A session failure may lose progress since the last saved output.


In [ ]:
from kodoom.translate.gate_review import compare_gate, review_summary

if RUN_GATE and BASELINE_GATE_BUNDLE:
    gate = compare_gate(gate, BASELINE_GATE_BUNDLE)
result = gate if RUN_GATE else preflight
print("Stage status:", result["status"])
print("Save this private output bundle:", result["bundle"])
print("Review counts:", review_summary(result["artifact_root"]))
print("Stop here for human review; save outputs and verify a fresh-session restore.")